# 04 - Analysis

Qualitative figures for individual scenarios: the alignment paths a system produced,
and how its error evolves over the course of a performance.

Requires the experiments and evaluation for `BENCHMARK` to have been run (notebooks 01-03,
or `python benchmark.py run`). Figures are written to `figures/`.

In [ ]:
import os
import pickle

import numpy as np
import plotly.graph_objs as go

from eval_tools import getGroundTruthTimestamps
from utils.constants import display_name

In [ ]:
BENCHMARK = "test"
SCENARIOS_DIR = {"train_small": "scenarios", "train": "scenarios_train", "test": "scenarios_test"}[BENCHMARK]
EXP_DIR = {"train_small": "experiments", "train": "experiments_train", "test": "experiments_test"}[BENCHMARK]
EVAL_DIR = {"train_small": "eval", "train": "eval_train", "test": "eval_test"}[BENCHMARK]

os.makedirs("figures", exist_ok=True)

## Alignment paths

Overlays each system's estimated alignment against the annotated beat pairs for one
scenario. Traces are added in reverse order of interest so the system under study is
drawn on top of the baselines it overlaps.

In [ ]:
s = 1
systems = ["MM_DIXON", "MM_ARZT", "SOA"]
colors = {"MM_DIXON": "blue", "MM_ARZT": "green", "SOA": "red"}

hyps = {system: np.load(f"{EXP_DIR}/{system}/s{s}/hyp.npy") for system in systems}
gt = getGroundTruthTimestamps(f"{SCENARIOS_DIR}/s{s}/query.beats",
                             f"{SCENARIOS_DIR}/s{s}/ref.beats").T

In [ ]:
fig = go.Figure()
fig.update_layout(
    xaxis_title='Query Time (s)',
    yaxis_title='Reference Time (s)',
    width=900,
    height=700,
    margin=dict(t=10, l=20, r=20, b=20),  # less whitespace on sides
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='Black',
        font=dict(size=20),  # big legend font
    ),
    font=dict(size=20),  # big overall font size for labels, ticks, etc.
    xaxis=dict(title_font=dict(size=24), tickfont=dict(size=16)),
    yaxis=dict(title_font=dict(size=24), tickfont=dict(size=16)),
)

for system in systems:
    fig.add_trace(go.Scatter(
        x=hyps[system][0], y=hyps[system][1], mode='lines',
        name=display_name(system), line=dict(color=colors[system]),
    ))
fig.add_trace(go.Scatter(
    x=gt[0], y=gt[1], mode='markers', name='Ground Truth',
    marker=dict(size=8, color='black'),
))
fig.write_image(f'figures/alignment_paths_s{s}.png', scale=2)
fig.show()

## Error over time

`errs.pkl` holds the signed error at every annotated beat, so plotting one scenario's
entry shows where a system drifts or recovers.

In [ ]:
s = 8
systems = ["DTW", "SOA"]

errs = {}
for system in systems:
    with open(f"{EVAL_DIR}/{system}/errs.pkl", "rb") as f:
        errs[system] = pickle.load(f)[f"s{s}"]

In [ ]:
fig = go.Figure()
fig.update_layout(
    title='Alignment Errors',
    xaxis_title='Annotated Beat',
    yaxis_title='Error (s)',
)
for system in systems:
    fig.add_trace(go.Scatter(
        x=np.arange(len(errs[system])), y=errs[system],
        mode='lines', name=display_name(system),
    ))
fig.write_image(f'figures/errors_over_time_s{s}.png', scale=2)
fig.show()